<a href="https://colab.research.google.com/github/Jimber120/DA-Project-Member-3/blob/main/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Grouped Split, Baseline Benchmarking & Model Training

**Project:** FlyRank Ranking Signal Analysis — Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 02 of 03 (Model Training & Benchmark Optimization)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
This notebook is your starting template for model training and optimization:
1. **Grouped Holdout Split Design:** Enforce a strict zero-client-leakage partition on `client_id` with programmatic disjointness assertions.
2. **Transparent Heuristic Baselines:** Fit Baseline 1 (Mean Dummy Regressor) and Baseline 2 (Univariate Word Count Rule) to establish the floor to beat.
3. **High-Accuracy Model Training:**
   - Fit candidate algorithms (e.g., cross-validated Polynomial Interaction Ridge, HistGradientBoosting Regressor, or Stacking Ensemble).
   - Maximize generalization accuracy on unseen client domains.
4. **Payload Export:** Export model predictions, test set ground truth, and metadata to `work/outputs/model_evaluation_payload.pkl` for evaluation in Notebook 03.

### Running in Google Colab
1. **Runtime:** a standard CPU runtime is enough. Run the cells top to bottom.
2. **Persistence:** the first code cell mounts Google Drive and keeps everything under `MyDrive/flyrank_ranking_signal/`. Notebooks 01 -> 02 -> 03 hand files to each other through that folder, so they still work if Colab recycles the runtime between notebooks.


In [1]:
# Colab Setup Cell: project folders + persistence (run this first in every notebook)
# ----------------------------------------------------------------------
import os, sys

USE_DRIVE = True                     # True  -> outputs persist in Google Drive (recommended: Colab runtimes get recycled)
                                     # False -> outputs live in /content and vanish when the runtime resets
PROJECT_DIRNAME = "flyrank_ranking_signal"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = f"/content/drive/MyDrive/{PROJECT_DIRNAME}"
elif IN_COLAB:
    ROOT = f"/content/{PROJECT_DIRNAME}"
else:                                # fallback so the notebook still runs outside Colab
    ROOT = os.path.abspath(PROJECT_DIRNAME)

OUT_DIR = os.path.join(ROOT, "work", "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
print("Project root :", ROOT)
print("Outputs dir  :", OUT_DIR)


Mounted at /content/drive
Project root : /content/drive/MyDrive/flyrank_ranking_signal
Outputs dir  : /content/drive/MyDrive/flyrank_ranking_signal/work/outputs


## 1. Grouped Holdout Split Design (Zero Client Leakage)

### 1.1 The Multi-Tenant Generalization Challenge
In multi-tenant SaaS platforms, standard random row splits produce severe **client leakage**: content items from the same client share identical domain authority, root backlink velocity, CMS infrastructure, and brand equity. A random split allows models to memorize client-level baseline volume, inflating test performance while collapsing on brand-new client domains.

### 1.2 The Client Holdout Standard
To guarantee 100% scientific validity:
* We partition strictly by `client_id` using `GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)`.
* We assert that training clients and test clients are mutually exclusive:
  $$\text{set}(\text{clients}_{\text{train}}) \cap \text{set}(\text{clients}_{\text{test}}) = \emptyset$$
* Test metrics reflect true generalization performance on entirely unseen enterprise domains.

In [2]:
# Code Cell 1: Load Features and Execute Grouped Split
# ----------------------------------------------------------------------
# 1a. Imports & paths
# ----------------------------------------------------------------------
import os, json, pickle, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:                       # scikit-learn < 1.4
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y, p): return float(np.sqrt(mean_squared_error(y, p)))

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
RANDOM_STATE = 42

# ROOT / OUT_DIR come from the Colab Setup Cell above (outputs of Notebook 01 live there)
if not (os.path.exists(os.path.join(OUT_DIR, "prepared_features.parquet")) or
        os.path.exists(os.path.join(OUT_DIR, "prepared_features.csv"))):
    raise FileNotFoundError(f"prepared_features.* not found in {OUT_DIR} — run Notebook 01 first "
                            "(and keep USE_DRIVE the same in all three notebooks).")

# ----------------------------------------------------------------------
# 1b. Load prepared features (parquet preferred, CSV fallback)
# ----------------------------------------------------------------------
pq, cs = (os.path.join(OUT_DIR, f"prepared_features.{e}") for e in ("parquet", "csv"))
try:
    data = pd.read_parquet(pq)
except Exception:
    data = pd.read_csv(cs)

META = ["content_id", "client_id", "target_log_impressions"]
X = data.drop(columns=META)
y = data["target_log_impressions"]
groups = data["client_id"]
ids = data["content_id"]
assert not X.isna().any().any(), "NaNs in feature matrix"
print("Feature matrix:", X.shape)

# ----------------------------------------------------------------------
# 1c. Grouped holdout split: zero client overlap
# ----------------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
clients_train, clients_test = groups.iloc[train_idx], groups.iloc[test_idx]
ids_test = ids.iloc[test_idx]

assert set(clients_train).isdisjoint(set(clients_test)), "Client leakage detected between train and test sets!"
print(f"Train Cohort: {len(X_train):,} rows across {clients_train.nunique()} clients")
print(f"Test Cohort : {len(X_test):,} rows across {clients_test.nunique()} unseen clients")
print(f"Target (train): mean={y_train.mean():.3f} std={y_train.std():.3f}")
print(f"Target (test) : mean={y_test.mean():.3f} std={y_test.std():.3f}   <- a mean shift here is why Baseline 1 R2 can be < 0")

Feature matrix: (30000, 49)
Train Cohort: 23,837 rows across 25 clients
Test Cohort : 6,163 rows across 7 unseen clients
Target (train): mean=6.309 std=2.677
Target (test) : mean=5.724 std=2.683   <- a mean shift here is why Baseline 1 R2 can be < 0


## 2. Transparent Heuristic Baseline Benchmarks

To verify that machine learning provides genuine predictive lift rather than fitting noise, evaluate two non-ML baselines on the exact same test cohort:
1. **Baseline 1 (Mean Dummy Regressor):** Always predicts the training set target mean $\bar{y}_{\text{train}}$. (Yields $R^2 = 0.0$ on train; negative $R^2$ on held-out clients with a lower mean).
2. **Baseline 2 (Univariate Word Count Rule):** A simple single-variable OLS regression on `log_word_count`, reflecting the common SEO belief that "longer articles rank higher and gain more visibility".

In [3]:
# Code Cell 2: Fit and Evaluate Standardized Heuristic Baselines
# ----------------------------------------------------------------------
# 2a. Evaluation helper
# ----------------------------------------------------------------------
def evaluate_predictions(y_true, y_pred, model_name):
    rho = spearmanr(y_true, y_pred)[0]
    return {"Model": model_name,
            "R2": r2_score(y_true, y_pred),
            "RMSE": root_mean_squared_error(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "Spearman_Rho": float(rho) if np.isfinite(rho) else 0.0}   # constant predictions => undefined rho

results = []

# ----------------------------------------------------------------------
# 2b. Baseline 1: Mean dummy (floor). Negative R2 on unseen clients is EXPECTED, not a bug.
# ----------------------------------------------------------------------
b1 = DummyRegressor(strategy="mean").fit(X_train, y_train)
y_pred_b1 = b1.predict(X_test)
results.append(evaluate_predictions(y_test, y_pred_b1, "Baseline 1 (Mean Dummy)"))

# ----------------------------------------------------------------------
# 2c. Baseline 2: Univariate OLS on log_word_count ("longer is better" editorial rule)
# ----------------------------------------------------------------------
b2 = LinearRegression().fit(X_train[["log_word_count"]], y_train)
y_pred_b2 = b2.predict(X_test[["log_word_count"]])
results.append(evaluate_predictions(y_test, y_pred_b2, "Baseline 2 (Word Count Rule)"))

print("BASELINE PERFORMANCE (held-out clients)")
display(pd.DataFrame(results))

BASELINE PERFORMANCE (held-out clients)


,Model,R2,RMSE,MAE,Spearman_Rho
0,Baseline 1 (Mean Dummy),-0.0476,2.7457,2.2757,0.0000
1,Baseline 2 (Word Count Rule),-0.0112,2.6976,2.2441,0.0429


## 3. High-Accuracy Model Training

### 3.1 Model Architecture Strategy
To achieve the highest honest accuracy without data leakage, consider:
1. **Regularized Linear Models with Interactions:** Pairwise interaction terms ($x_i \times x_j$) across primary signals regularized with `RidgeCV`.
2. **Gradient Boosted Decision Trees (`HistGradientBoostingRegressor`):** Captures sharp non-linear thresholds, step-function jumps, and missingness natively.
3. **Champion Blended Ensemble:** Blending linear macro-trends with localized tree splits ($w \cdot \hat{y}_{\text{Ridge}} + (1-w) \cdot \hat{y}_{\text{HGBR}}$) to maximize variance explained.

In [4]:
# Code Cell 3: Machine Learning Model Training & Optimization
# ----------------------------------------------------------------------
# 3a. Candidate A: Polynomial-interaction Ridge, alpha chosen with GroupKFold over CLIENTS
# ----------------------------------------------------------------------
PRIMARY = [c for c in ["serp_rank_power", "log_search_volume", "log_word_count",
                       "log_internal_links_in", "log_external_backlinks", "log_sessions_90d"]
           if c in X_train.columns]
REST = [c for c in X_train.columns if c not in PRIMARY]
print("Primary signals for interactions:", PRIMARY)

preprocess = ColumnTransformer([
    ("primary_poly", Pipeline([("scale", StandardScaler()),
                               ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False))]), PRIMARY),
    ("rest", StandardScaler(), REST),
])
Xtr_t = preprocess.fit_transform(X_train)          # fit on TRAIN only
Xte_t = preprocess.transform(X_test)

cv_splits = list(GroupKFold(n_splits=5).split(Xtr_t, y_train, groups=clients_train))
ridge = RidgeCV(alphas=np.logspace(-2, 3, 20), cv=cv_splits).fit(Xtr_t, y_train)
y_pred_ridge = ridge.predict(Xte_t)
print(f"Ridge: selected alpha = {ridge.alpha_:.4g}, design matrix width = {Xtr_t.shape[1]}")
results.append(evaluate_predictions(y_test, y_pred_ridge, "Ridge Interactions"))

# ----------------------------------------------------------------------
# 3b. Candidate B: HistGradientBoostingRegressor
# ----------------------------------------------------------------------
hgbr = HistGradientBoostingRegressor(
    max_iter=250, learning_rate=0.06, max_leaf_nodes=31,
    min_samples_leaf=20, l2_regularization=1.5, random_state=RANDOM_STATE,
)
hgbr.fit(X_train, y_train)
y_pred_hgbr = hgbr.predict(X_test)
results.append(evaluate_predictions(y_test, y_pred_hgbr, "HistGradientBoosting"))

# ----------------------------------------------------------------------
# 3c. Champion: convex blend. Weights are fixed a priori — tuning them on the test cohort would leak.
#     Template default = 0.50/0.50. The master guide uses 0.40 Ridge / 0.60 HGBR; change W_RIDGE to switch.
# ----------------------------------------------------------------------
W_RIDGE = 0.50
W_HGBR = 1.0 - W_RIDGE
y_pred_ensemble = W_RIDGE * y_pred_ridge + W_HGBR * y_pred_hgbr
results.append(evaluate_predictions(y_test, y_pred_ensemble, "Champion Stacking Ensemble"))

res = pd.DataFrame(results)
res["RMSE_Reduction_%"] = 100 * (res.loc[0, "RMSE"] - res["RMSE"]) / res.loc[0, "RMSE"]
print("\nMODEL COMPARISON (held-out clients)")
display(res)

# ----------------------------------------------------------------------
# 3d. What is the tree model using? (permutation importance on held-out clients)
# ----------------------------------------------------------------------
from sklearn.inspection import permutation_importance
pi = permutation_importance(hgbr, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
print("\nTop 12 features by permutation importance (drop in R2):")
display(pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False).head(12).to_frame("importance"))

Primary signals for interactions: ['serp_rank_power', 'log_search_volume', 'log_word_count', 'log_sessions_90d']
Ridge: selected alpha = 1000, design matrix width = 55

MODEL COMPARISON (held-out clients)


,Model,R2,RMSE,MAE,Spearman_Rho,RMSE_Reduction_%
0,Baseline 1 (Mean Dummy),-0.0476,2.7457,2.2757,0.0000,0.0000
1,Baseline 2 (Word Count Rule),-0.0112,2.6976,2.2441,0.0429,1.7510
2,Ridge Interactions,0.3818,2.1092,1.7983,0.6046,23.1821
3,HistGradientBoosting,0.5323,1.8346,1.4751,0.7212,33.1833
4,Champion Stacking Ensemble,0.4916,1.9128,1.6046,0.7006,30.3331



Top 12 features by permutation importance (drop in R2):


,importance
log_pageviews_90d,0.3101
log_sessions_90d,0.0691
log_days_since_last_update,0.0493
log_content_age_days,0.0342
log_users_90d,0.0329
engagement_volume_synergy,0.0325
imputed_avg_position,0.0232
log_word_count,0.0194
char_count,0.0160
scroll_per_pageview,0.0081


In [5]:
# Code Cell 4: Export Model Evaluation Payload
# ----------------------------------------------------------------------
# 4a. Pickle payload for Notebook 03
# ----------------------------------------------------------------------
preds = {
    "Baseline 1 (Mean Dummy)": y_pred_b1,
    "Baseline 2 (Word Count Rule)": y_pred_b2,
    "Ridge Interactions": y_pred_ridge,
    "HistGradientBoosting": y_pred_hgbr,
    "Champion Stacking Ensemble": y_pred_ensemble,
}
payload = {
    "X_test": X_test,
    "y_test": y_test,
    "clients_test": clients_test,
    "ids_test": ids_test,
    "preds": preds,
    "ensemble_weights": {"ridge": W_RIDGE, "hgbr": W_HGBR},
}
payload_path = os.path.join(OUT_DIR, "model_evaluation_payload.pkl")
with open(payload_path, "wb") as f:
    pickle.dump(payload, f)

# ----------------------------------------------------------------------
# 4b. Flat predictions CSV (ids + truth + every model)
# ----------------------------------------------------------------------
pred_df = pd.DataFrame({"content_id": ids_test.values, "client_id": clients_test.values,
                        "y_true": y_test.values})
for name, p in preds.items():
    pred_df[name] = p
csv_path = os.path.join(OUT_DIR, "model_test_predictions.csv")
pred_df.to_csv(csv_path, index=False)

# ----------------------------------------------------------------------
# 4c. Confirmation
# ----------------------------------------------------------------------
for p in (payload_path, csv_path):
    print(f"{os.path.relpath(p, ROOT):55s} {os.path.getsize(p)/1024:,.1f} KB")
print("Evaluation payload successfully exported.")

work/outputs/model_evaluation_payload.pkl               3,139.3 KB
work/outputs/model_test_predictions.csv                 887.4 KB
Evaluation payload successfully exported.
